# Ch.2 — Isolation Forest

> Anomalies are few and different — they require fewer random splits to isolate. No distribution assumptions, no density estimation. Just trees.

**Dataset:** Credit Card Fraud Detection — 284,807 transactions, 0.17% fraud  
**Task:** Score transactions by isolation path length, target ~72% recall @ 0.5% FPR

## The Core Idea

Isolation Forest isolates observations by randomly selecting features and split values. The anomaly score:

$$s(\mathbf{x}, n) = 2^{-E[h(\mathbf{x})]/c(n)}$$

- $h(\mathbf{x})$ = path length (splits to isolate)
- $c(n)$ = average path length in a BST with $n$ nodes
- $s \to 1$: anomaly (short path) | $s \to 0.5$: normal (average path)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
IMG_DIR = "img/"

import os
os.makedirs(IMG_DIR, exist_ok=True)
print("Libraries loaded.")

In [ ]:
# ── Load and split ─────────────────────────────────────────────────────────
df = pd.read_csv("creditcard.csv")
if os.environ.get("CI"):  # limit rows for CI runs
    df = df.head(1000)

X = df.drop("Class", axis=1).values
y = df["Class"].values
feature_names = [c for c in df.columns if c != "Class"]

split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train: {X_train.shape} ({y_train.sum()} fraud)")
print(f"Test:  {X_test.shape} ({y_test.sum()} fraud)")
print(f"Fraud rate: {y.mean():.4%}")

## Training the Isolation Forest

Key hyperparameters:
- `n_estimators`: number of trees (more = more stable scores)
- `max_samples`: sub-sample size per tree (256 is the paper's default)
- `contamination`: expected fraction of anomalies (NOT the default 0.5!)

In [ ]:
# ── Train Isolation Forest ─────────────────────────────────────────────────
iso_forest = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination=0.002,
    random_state=SEED,
    n_jobs=-1,
)
iso_forest.fit(X_train)

# Score test set (sklearn returns negative scores; flip so higher = more anomalous)
raw_scores = iso_forest.decision_function(X_test)
scores_if = -raw_scores

print(f"Score range: [{scores_if.min():.4f}, {scores_if.max():.4f}]")
print(f"Mean score (legit): {scores_if[y_test == 0].mean():.4f}")
print(f"Mean score (fraud): {scores_if[y_test == 1].mean():.4f}")

In [ ]:
# ── ROC Curve and Recall @ 0.5% FPR ────────────────────────────────────────
fpr_if, tpr_if, thresh_if = roc_curve(y_test, scores_if)
auc_if = auc(fpr_if, tpr_if)

idx_005 = np.where(fpr_if <= 0.005)[0][-1]
recall_if = tpr_if[idx_005]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr_if, tpr_if, color="#e67e22", linewidth=2, label=f"IF (AUC={auc_if:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].scatter([fpr_if[idx_005]], [tpr_if[idx_005]], color="#e74c3c", s=100, zorder=5,
               label=f"@ 0.5% FPR: {recall_if:.1%}")
axes[0].set(title="ROC — Isolation Forest", xlabel="FPR", ylabel="Recall")
axes[0].legend()

# Score distribution
fraud_mask = y_test == 1
axes[1].hist(scores_if[~fraud_mask], bins=100, alpha=0.6, color="#27ae60",
            label="Legitimate", density=True)
axes[1].hist(scores_if[fraud_mask], bins=50, alpha=0.7, color="#e74c3c",
            label="Fraud", density=True)
axes[1].set(title="Anomaly Score Distribution", xlabel="Score", ylabel="Density")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{IMG_DIR}roc_isolation_forest.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nRecall @ 0.5% FPR: {recall_if:.2%} (vs Z-score ~45%)")

## Hyperparameter Sensitivity

How do `n_estimators` and `max_samples` affect performance?

In [ ]:
# ── n_estimators sweep ─────────────────────────────────────────────────────
n_trees_list = [10, 50, 100, 200, 500]
results_trees = []

for n_trees in n_trees_list:
    model = IsolationForest(n_estimators=n_trees, max_samples=256,
                           contamination=0.002, random_state=SEED, n_jobs=-1)
    model.fit(X_train)
    scores = -model.decision_function(X_test)
    fpr_i, tpr_i, _ = roc_curve(y_test, scores)
    auc_i = auc(fpr_i, tpr_i)
    idx = np.where(fpr_i <= 0.005)[0][-1]
    results_trees.append({"n_trees": n_trees, "AUC": auc_i, "Recall@0.5%FPR": tpr_i[idx]})

res_df = pd.DataFrame(results_trees)
print(res_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(res_df["n_trees"], res_df["Recall@0.5%FPR"], "o-", color="#e67e22", linewidth=2)
ax.set(title="Effect of n_estimators on Recall", xlabel="Number of Trees",
       ylabel="Recall @ 0.5% FPR")
ax.axhline(0.80, color="#e74c3c", linestyle="--", alpha=0.5, label="Target: 80%")
ax.legend()
plt.tight_layout()
plt.savefig(f"{IMG_DIR}n_estimators_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Manual Isolation Tree (educational) ────────────────────────────────────
def build_itree(X, depth=0, max_depth=8):
    """Build one isolation tree from scratch."""
    n, d = X.shape
    if n <= 1 or depth >= max_depth:
        return {"type": "leaf", "size": n, "depth": depth}
    q = np.random.randint(d)  # random feature
    p = np.random.uniform(X[:, q].min(), X[:, q].max())  # random split
    left_mask = X[:, q] < p
    return {
        "type": "split", "feature": q, "threshold": p,
        "left": build_itree(X[left_mask], depth + 1, max_depth),
        "right": build_itree(X[~left_mask], depth + 1, max_depth),
    }

def path_length(x, tree, depth=0):
    """Return path length for a single point."""
    if tree["type"] == "leaf":
        return depth
    if x[tree["feature"]] < tree["threshold"]:
        return path_length(x, tree["left"], depth + 1)
    return path_length(x, tree["right"], depth + 1)

# Build 10 trees on a small sample
np.random.seed(SEED)
sample_idx = np.random.choice(len(X_train), 256, replace=False)
X_sample = X_train[sample_idx]

trees = [build_itree(X_sample) for _ in range(10)]

# Compare path lengths for a fraud vs normal transaction
fraud_idx = np.where(y_test == 1)[0][0]
normal_idx = np.where(y_test == 0)[0][0]

paths_fraud = [path_length(X_test[fraud_idx], t) for t in trees]
paths_normal = [path_length(X_test[normal_idx], t) for t in trees]

print(f"Fraud transaction — path lengths:  {paths_fraud}, mean={np.mean(paths_fraud):.1f}")
print(f"Normal transaction — path lengths: {paths_normal}, mean={np.mean(paths_normal):.1f}")
print(f"\nFraud is isolated faster (shorter paths) → higher anomaly score")

In [ ]:
# ── Feature importance via split frequency ─────────────────────────────────
# Which features does Isolation Forest use most for splits?
feature_counts = np.zeros(X_train.shape[1])
for tree in iso_forest.estimators_:
    features_used = tree.tree_.feature
    for f in features_used:
        if f >= 0:  # -2 means leaf node
            feature_counts[f] += 1

feature_counts /= feature_counts.sum()
top_idx = np.argsort(feature_counts)[-10:][::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([feature_names[i] for i in top_idx], feature_counts[top_idx], color="#e67e22")
ax.set(title="Top 10 Features by Split Frequency (Isolation Forest)",
       xlabel="Fraction of Splits")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{IMG_DIR}feature_importance_if.png", dpi=150, bbox_inches="tight")
plt.show()

## Progress Check

| Constraint | Target | Status |
|------------|--------|--------|
| #1 DETECTION | >80% recall | ❌ ~72% — better but not there |
| #2 PRECISION | <0.5% FPR | ✅ ROC thresholding |
| #3 REAL-TIME | <100ms | ✅ ~5ms inference |
| #4 ADAPTABILITY | Drift handling | ❌ Static model |
| #5 EXPLAINABILITY | Justifiable | ⚡ Feature importance available |

**Next**: Ch.3 — Autoencoders (learn what normal looks like, ~78% recall)

## Exercises

In [ ]:
# ── Exercise 1: max_samples sweep ──────────────────────────────────────────
# TODO: Test max_samples in [64, 128, 256, 512, 1024]
# For each, train IF with 200 trees, compute recall @ 0.5% FPR
# Plot the results. Does larger max_samples always help?

pass

In [ ]:
# ── Exercise 2: Feature subset training ────────────────────────────────────
# TODO: Train IF using only the top 5 most discriminative features
# (from Ch.1 analysis: V14, V17, V12, V10, V16)
# Compare recall vs full-feature IF. Does feature selection help?

pass

In [ ]:
# ── Exercise 3: Contamination sensitivity ──────────────────────────────────
# TODO: Test contamination in [0.0005, 0.001, 0.002, 0.005, 0.01, 0.05]
# For each, compute both sklearn's default threshold AND ROC-based threshold
# Show that ROC-based thresholding is more robust to contamination choice

pass

## Additional Visualization — Precision-Recall Perspective

ROC can look optimistic on heavily imbalanced datasets. Plot PR behavior to inspect precision-recall trade-offs directly.

In [ ]:
# ── Precision-Recall curve and threshold diagnostics ─────────────────────────
precision_if, recall_curve_if, pr_thresh_if = precision_recall_curve(y_test, scores_if)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(recall_curve_if, precision_if, color="#8e44ad", linewidth=2)
axes[0].set(title="Precision-Recall Curve — Isolation Forest", xlabel="Recall", ylabel="Precision")
axes[0].axhline(y=y_test.mean(), color="#7f8c8d", linestyle="--", alpha=0.6, label="Fraud base rate")
axes[0].legend()

if len(pr_thresh_if) > 0:
    sample_idx = np.linspace(0, len(pr_thresh_if) - 1, num=min(15, len(pr_thresh_if)), dtype=int)
    sampled_t = pr_thresh_if[sample_idx]
    sampled_r = recall_curve_if[:-1][sample_idx]
    sampled_p = precision_if[:-1][sample_idx]
    axes[1].plot(sampled_t, sampled_r, "o-", label="Recall", color="#16a085")
    axes[1].plot(sampled_t, sampled_p, "o-", label="Precision", color="#c0392b")
    axes[1].set(title="Threshold vs Precision/Recall", xlabel="Anomaly score threshold")
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "Not enough threshold points", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.savefig(f"{IMG_DIR}pr_isolation_forest.png", dpi=150, bbox_inches="tight")
plt.show()

## Additional Visualization — Score Quantiles by Class

Inspect quantiles to understand overlap between legitimate and fraud score distributions.

In [ ]:
# ── Class-wise score quantiles and overlap window ───────────────────────────
q_levels = [0.50, 0.75, 0.90, 0.95, 0.99]
q_legit = np.quantile(scores_if[y_test == 0], q_levels)
q_fraud = np.quantile(scores_if[y_test == 1], q_levels)

quant_df = pd.DataFrame({
    "quantile": q_levels,
    "legitimate": q_legit,
    "fraud": q_fraud,
    "gap(fraud-legit)": q_fraud - q_legit,
})
print(quant_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(q_levels, q_legit, "o-", label="Legitimate", color="#27ae60")
ax.plot(q_levels, q_fraud, "o-", label="Fraud", color="#e74c3c")
ax.set(title="Isolation Score Quantiles by Class", xlabel="Quantile", ylabel="Anomaly score")
ax.legend()
plt.tight_layout()
plt.savefig(f"{IMG_DIR}quantiles_isolation_forest.png", dpi=150, bbox_inches="tight")
plt.show()